In [4]:
import sqlite3

# 데이터베이스 연결 및 테이블 생성 함수 정의
def initialize_database():
    # SQLite 데이터베이스 파일에 연결 (없는 경우 새로 생성)
    conn = sqlite3.connect('withpaw.db')
    cursor = conn.cursor()

    # 사용자 프로필 테이블 생성
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS user_profile (
        id INTEGER PRIMARY KEY AUTOINCREMENT,    -- 사용자 ID (자동 증가)
        name TEXT,                               -- 사용자 이름
        email TEXT                               -- 사용자 이메일 (선택)
    )
    """)

    # 반려동물 정보 테이블 생성
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS pets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,   -- 반려동물 ID (자동 증가)
        name TEXT NOT NULL,                     -- 반려동물 이름
        age INTEGER,                            -- 반려동물 나이
        species TEXT,                           -- 반려동물 종 (예: 강아지, 고양이)
        breed TEXT,                             -- 반려동물 품종
        family_id INTEGER,                      -- 반려동물 가족 ID (user_profile와 연결)
        FOREIGN KEY(family_id) REFERENCES user_profile(id)   -- 외래 키 설정
    )
    """)

    print("데이터베이스 및 테이블 생성 완료!")
    conn.commit()  # 변경 사항 저장
    conn.close()   # 데이터베이스 연결 종료

# 사용자 프로필 설정 함수 정의
def set_user_profile():
    conn = sqlite3.connect("withpaw.db")     # 데이터베이스 연결
    cursor = conn.cursor()

    # 사용자 이름과 이메일 입력 받기
    name = input("사용자 이름을 입력하세요: ")
    email = input("사용자 이메일을 입력하세요 (선택): ")

    # 사용자 정보를 테이블에 삽입
    cursor.execute("INSERT INTO user_profile (name, email) VALUES (?, ?)", (name, email))
    conn.commit()   # 변경 사항 저장
    conn.close()    # 데이터베이스 연결 종료
    print(f"프로필이 저장되었습니다! 이름: {name}, 이메일: {email}")

# 반려동물 등록 함수 정의
def add_pet():
    conn = sqlite3.connect("withpaw.db")    # 데이터베이스 연결
    cursor = conn.cursor()

    # 사용자 목록 출력 (반려동물 가족 선택을 위해)
    cursor.execute("SELECT id, name FROM user_profile")
    users = cursor.fetchall()
    if not users:   # 등록된 사용자가 없을 경우
        print("먼저 사용자 프로필을 생성해주세요.")
        conn.close()
        return
    
    print("\n등록된 사용자: ")
    for user in users:
        print(f"ID: {user[0]}, 이름: {user[1]}")

    # 반려동물 가족 ID 입력 받기
    try:
        family_id = int(input("반려동물의 가족 ID를 입력하세요: "))
        if not any(user[0] == family_id for user in users):
            print("잘못된 가족 ID입니다. 다시 시도해주세요.")
            conn.close()
            return
    except ValueError:
        print("유효한 숫자를 입력하세요.")
        conn.close()
        return

    # 반려동물 정보 입력 받기
    name = input("반려동물 이름을 입력하세요: ")
    age = int(input("반려동물 나이를 입력하세요: "))
    species = input("반려동물 종을 입력하세요 (예: 강아지, 고양이): ")
    breed = input("반려동물 품종을 입력하세요: ")

    # 반려동물 정보를 테이블에 삽입
    cursor.execute("""
    INSERT INTO pets (name, age, species, breed, family_id)
    VALUES (?, ?, ?, ?, ?)
    """, (name, age, species, breed, family_id))
    conn.commit()  # 변경 사항 저장
    conn.close()   # 데이터베이스 연결 종료
    print(f"{name}이(가) 성공적으로 등록되었습니다!")

# 사용자와 반려동물 데이터 조회 함수 정의
def show_data():
    conn = sqlite3.connect("withpaw.db")    # 데이터베이스 연결
    cursor = conn.cursor()

    # 사용자 프로필 데이터 출력
    print("\n사용자 목록:")
    cursor.execute("SELECT * FROM user_profile")
    for row in cursor.fetchall():
        print(row)

    # 반려동물 데이터 출력 (가족 정보 포함)
    print("\n반려동물 목록:")
    cursor.execute("""
    SELECT pets.name, pets.age, pets.species, pets.breed, user_profile.name AS family
    FROM pets
    JOIN user_profile ON pets.family_id = user_profile.id
    """)
    for row in cursor.fetchall():
        print(f"이름: {row[0]}, 나이: {row[1]}, 종: {row[2]}, 품종: {row[3]}, 가족: {row[4]}")

    conn.close()    # 데이터베이스 연결 종료

# 메인 실행 흐름
if __name__ == "__main__":
    initialize_database()   # 데이터베이스 초기화

    while True:
        print("\n1. 사용자 프로필 설정")
        print("2. 반려동물 등록")
        print("3. 데이터 조회")
        print("4. 종료")

        choice = input("선택: ")
        if choice == "1":
            set_user_profile()
        elif choice == "2":
            add_pet()
        elif choice == "3":
            show_data()
        elif choice == "4":
            print("위드포를 종료합니다. 감사합니다!")
            break
        else:
            print("잘못된 입력입니다. 다시 선택해주세요!")

OperationalError: unrecognized token: "#"